# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, overview, and explore the [FAIR^2 Croissant dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

The Croissant schema provides a structured metadata description for this multi-table dataset.

### Dataset Source
The dataset's Croissant schema is available at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed (will be a no-op if already present)
!pip install -q mlcroissant

## 1. Data Loading
We'll load the Croissant metadata and quickly verify key properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

# Convert to JSON to pretty-print the dataset info
metadata = dataset.metadata.to_json()
print(f"Dataset name: {getattr(dataset.metadata, 'name', None)}\n")
print(f"Dataset description: {getattr(dataset.metadata, 'description', None)}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', None)}\n")
print(f"Version: {getattr(dataset.metadata, 'version', None)}\n")

## 2. Data Overview
Let's discover available record sets and their fields using `mlcroissant` meta-API.

*All entities are referenced via their `@id` fields as per FAIR best practices and Croissant conventions.*

In [ ]:
# List all record sets
print("Available record sets with @id (and name/description):\n")
record_sets = [(rs['@id'], rs.get('name', ''), rs.get('description', '')) for rs in dataset.metadata.to_json().get('recordSet', [])]
for rec_id, rec_name, rec_desc in record_sets:
    print(f"@id: {rec_id}\n  name: {rec_name}\n  description: {rec_desc}\n")

# Example: List all fields for the first record set
if record_sets:
    first_record_set_id = record_sets[0][0]
    print(f"Fields for record set {first_record_set_id}:")
    rs_meta = None
    for rs in dataset.metadata.to_json()['recordSet']:
        if rs['@id'] == first_record_set_id:
            rs_meta = rs
            break
    if rs_meta:
        for field in rs_meta.get('field', []):
            fobj = field if isinstance(field, dict) else {}
            # Could be string @id or inline dict
            if isinstance(field, dict):
                fid = field.get('@id')
                name = field.get('name', '')
                dtype = field.get('dataType', '')
            else:
                fid = field
                name = ''
                dtype = ''
            print(f"  Field @id: {fid}\n    name: {name}\n    dtype: {dtype}")

## 3. Data Extraction
Let's load tabular data from a specific record set and display its columns and a sample of records.

All references use the `@id` of target record sets and fields.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print(f"Record set @id list: {record_set_ids}\n")

# For demonstration, extract first (main) record set, or adjust as needed
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

# If at least one record set, display its shape and top 5 rows
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"First record set ({first_id}) head:")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate some common EDA steps using column/field `@id`s. These include filtering, normalization, and grouping. 

> ⚠️ Update the example code below to use the desired numeric and group fields from output above, using their `@id` as the DataFrame column name.

In [ ]:
# -- Example EDA with placeholders; refer to previous outputs to identify actual @id field names --
# Suppose first record set contains an abundance field with @id 'cr:abundance' and a group field 'cr:group'

record_set_id = record_set_ids[0] if record_set_ids else None

# List numeric columns and select one for illustration
df = dataframes[record_set_id]
print("Numeric fields in the DataFrame:")
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(numeric_fields)

# Pick the first numeric field for demo (replace with target @id if desired)
if numeric_fields:
    numeric_field = numeric_fields[0]

    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as an example
    print(f"Filtering records where {numeric_field} > {threshold:.2f}")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/object column
    group_fields = df.select_dtypes(include='object').columns.tolist()
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields in this record set.")

## 5. Visualization
Let's plot the distribution of the chosen numeric field and the normalized values. You can further extend by making boxplots, pairplots, or group-wise visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Histogram of original field
    sns.histplot(df[numeric_field].dropna(), bins=30, ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field}")

    # After normalization (if computed)
    if f"{numeric_field}_normalized" in filtered_df.columns:
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, ax=axes[1], kde=True)
        axes[1].set_title(f"Normalized {numeric_field} in filtered subset")
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No suitable field for visualization.")

## 6. Conclusion
Using the `mlcroissant` library, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset, reviewed its schema (record sets and fields using their `@id`), extracted tabular data, filtered and normalized relevant fields, and visualized distributions. This establishes a robust pipeline for FAIR, schema-centric multi-table data exploration.

To extend: change the field `@id`s and analysis steps to match analytical needs, or combine with domain-specific protein/omics downstream analysis.
